In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import joblib
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('Cellphone.csv')
df.head()

In [ ]:
# removing irrelevant columns
df.drop(['Product_id'], axis=1, inplace=True)

In [ ]:
df.info()

In [ ]:
df.rename(columns={'cpu core': 'cpu_core', 'cpu freq': 'cpu_freq', 'internal mem': 'internal_mem'}, inplace=True)

In [ ]:
df['Price'] = df['Price'].astype('float64')

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
#df.drop_duplicates(inplace=True)

In [ ]:
df.describe().T

In [ ]:
#df.describe().T

In [ ]:
# plotting boxplots for independent variables to check for outliers
numeric_cols = df.select_dtypes(include=np.number).columns
n = len(numeric_cols)
ncols = 4
nrows = int(np.ceil(n / ncols))

fig, axes_box = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5 * ncols, 4 * nrows))
axes_box = axes_box.flatten()

for ax, col in zip(axes_box, numeric_cols):
    sns.boxplot(x=df[col], ax=ax)
    ax.set_title(col)

for ax in axes_box[n:]:
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
def Handle_outliers(p_df, column):
    q1 = p_df[column].quantile(0.25)
    q3 = p_df[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)

    if col == 'Sale':
        lower_bound = 0
    elif col == 'ppi':
        lower_bound = 120
    elif (col == 'Front_Cam') or (col == 'RearCam'):
        lower_bound = 0
    elif (col == 'internal_mem') or (col == 'ram'):
        lower_bound = 0.004
    elif col == 'cpu_core':
        lower_bound = 1

    print(f"Column: {column}, Lower Bound: {lower_bound}, Upper Bound: {upper_bound}")
    p_df[column] = np.where(p_df[column] < lower_bound, lower_bound,np.where(p_df[column] > upper_bound, upper_bound, p_df[column]))

for col in df.columns:
    Handle_outliers(df, col)
    
        

In [ ]:
# plotting boxplots for independent variables post handling outliers
numeric_cols = df.select_dtypes(include=np.number).columns
n = len(numeric_cols)
ncols = 4
nrows = int(np.ceil(n / ncols))

fig, axes_box = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5 * ncols, 4 * nrows))
axes_box = axes_box.flatten()

for ax, col in zip(axes_box, numeric_cols):
    sns.boxplot(x=df[col], ax=ax)
    ax.set_title(col)

for ax in axes_box[n:]:
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# distribution of independent variables post handling outliers
numeric_cols = df.select_dtypes(include=np.number).columns

fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(15, 12))
for ax, col in zip(axes.flat, numeric_cols):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(col)

plt.tight_layout()
plt.show()

In [ ]:
# checking skewness of the data
df.skew()

In [ ]:
# applying transformations to reduce skewness
df_transformed = df.copy()
df_transformed['Sale_log'] = np.log1p(df_transformed['Sale'])
df_transformed['internal_mem_log'] = np.log1p(df_transformed['internal_mem'])
df_transformed['ram_sqrt'] = np.sqrt(df_transformed['ram'])
df_transformed['Front_Cam_log'] = np.log1p(df_transformed['Front_Cam'])
df_transformed['thickness_sqrt'] = np.sqrt(df_transformed['thickness'])
df_transformed.skew()


In [ ]:
#removed original columns after transformation
df_transformed.drop(['Sale','internal_mem', 'ram', 'Front_Cam', 'thickness'], axis=1, inplace=True)


In [ ]:
# plotting histograms for independent variables post transformation

cols = df_transformed.select_dtypes(include=np.number).columns
n = len(cols)
ncols = 3
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(5*ncols, 4*nrows))
axes = axes.flatten()

for ax, col in zip(axes, cols):
    sns.histplot(df_transformed[col], kde=True, ax=ax)
    ax.set_title(col)

for ax in axes[n:]:
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
corr = df_transformed.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.show()

In [ ]:
# preparing data for modeling
x = df_transformed.drop('Price', axis=1)
y = df_transformed['Price']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [ ]:
def model_metrics(model, x_tr, y_tr, x_te, y_te):
    model.fit(x_tr, y_tr)
    y_tr_pred = model.predict(x_tr)
    y_te_pred = model.predict(x_te)
    return {
        "RMSE Train": np.sqrt(mean_squared_error(y_tr, y_tr_pred)),
        "RMSE Test": np.sqrt(mean_squared_error(y_te, y_te_pred)),
        "R2 Train": r2_score(y_tr, y_tr_pred),
        "R2 Test": r2_score(y_te, y_te_pred),
    }

models_to_compare = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=1.0),
    "ElasticNet": ElasticNet(alpha=1.0, l1_ratio=0.5),
}

comparison = []

for model_name, model in models_to_compare.items():
    for scale_name, (X_tr, X_te) in [
        ("unscaled", (x_train, x_test)),
        ("scaled", (x_train_scaled, x_test_scaled))
    ]:
        metrics = model_metrics(model, X_tr, y_train, X_te, y_test)
        comparison.append({
            "Model": model_name,
            "Scaled": scale_name,
            **metrics
        })

performance_df = pd.DataFrame(comparison)
performance_df = performance_df.set_index(["Model", "Scaled"])
performance_df

In [ ]:
#dump the best performing model
joblib.dump(models_to_compare['LinearRegression'], 'linear_regression_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

In [ ]:
df_transformed.info()